## Data Overview & Platform Comparison

### By:
MGO

### Date:
2026-03-24

### Description:

Exploratory analysis of all collected data -- volume, quality, platform differences.
This notebook provides a comprehensive view of the collected corpus across all sources
(RSS, GDELT, ACLED). It also compares platform
characteristics and assesses data quality before feeding into the analysis pipeline.

## Import libraries

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

from utils.normalize import normalize_post

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Load all collected data

Load data from `data/01_raw/` for all sources. Normalize each source to the common
schema and combine into a single DataFrame. If raw data is not available, fall back
to sample data.

In [ ]:
data_dir = Path("../../data")
raw_dir = data_dir / "01_raw"
sample_path = data_dir / "sample" / "sample_posts.json"

all_posts = []

# Load RSS articles
rss_dir = raw_dir / "rss"
if rss_dir.exists():
    for f in sorted(rss_dir.glob("*.json")):
        with open(f) as fh:
            articles = json.load(fh)
        all_posts.extend([normalize_post(a, source="rss") for a in articles])
        print(f"  RSS: loaded {len(articles)} articles from {f.name}")

# Load GDELT articles
gdelt_dir = raw_dir / "gdelt"
if gdelt_dir.exists():
    for f in sorted(gdelt_dir.glob("*.json")):
        with open(f) as fh:
            articles = json.load(fh)
        all_posts.extend([normalize_post(a, source="gdelt") for a in articles])
        print(f"  GDELT: loaded {len(articles)} articles from {f.name}")

# Fallback to sample data if no raw data found
if not all_posts and sample_path.exists():
    print("No raw data found -- loading sample data for demonstration")
    with open(sample_path) as f:
        all_posts = json.load(f)

print(f"\nTotal posts loaded: {len(all_posts)}")

In [ ]:
# Build DataFrame and add derived columns
df = pd.DataFrame(all_posts)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
df["date"] = df["timestamp"].dt.date

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Platforms: {df['platform'].unique().tolist()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

In [ ]:
# Store in DuckDB for analytical queries
con = duckdb.connect()
con.register("posts", df)

# Quick SQL summary
summary = con.execute("""
    SELECT
        platform,
        COUNT(*) AS post_count,
        MIN(timestamp) AS earliest,
        MAX(timestamp) AS latest,
        ROUND(AVG(text_length), 0) AS avg_text_length,
        COUNT(DISTINCT source) AS unique_sources
    FROM posts
    GROUP BY platform
    ORDER BY post_count DESC
""").fetchdf()

print("Platform summary (via DuckDB):")
display(summary)

## Overall Statistics

In [ ]:
# Total posts per source
print("Posts per source:")
print(df["source"].value_counts().to_string())
print(f"\nTotal unique sources: {df['source'].nunique()}")

In [ ]:
# Volume over time (line chart)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Posts by platform (bar)
df["platform"].value_counts().plot(kind="bar", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Total Posts by Platform")
axes[0, 0].set_ylabel("Count")

# Top 10 sources (horizontal bar)
df["source"].value_counts().head(10).plot(kind="barh", ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Top 10 Sources")
axes[0, 1].set_xlabel("Count")

# Daily volume over time
daily_volume = df.set_index("timestamp").resample("D").size()
daily_volume.plot(ax=axes[1, 0], marker="o", linewidth=2, color="green")
axes[1, 0].set_title("Daily Post Volume")
axes[1, 0].set_ylabel("Posts")
axes[1, 0].set_xlabel("Date")

# Volume by platform over time
for platform in df["platform"].unique():
    platform_daily = df[df["platform"] == platform].set_index("timestamp").resample("D").size()
    platform_daily.plot(ax=axes[1, 1], label=platform, marker=".", linewidth=1.5)
axes[1, 1].set_title("Daily Volume by Platform")
axes[1, 1].set_ylabel("Posts")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Platform Comparison

Compare characteristics across platforms: sentiment tone, topic coverage,
volume patterns, and content length distributions.

In [ ]:
# Content length distributions by platform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="platform", y="text_length", ax=axes[0], palette="Set2")
axes[0].set_title("Text Length Distribution by Platform")
axes[0].set_ylabel("Characters")

sns.boxplot(data=df, x="platform", y="word_count", ax=axes[1], palette="Set2")
axes[1].set_title("Word Count Distribution by Platform")
axes[1].set_ylabel("Words")

plt.tight_layout()
plt.show()

# Statistics table
print("\nText length statistics by platform:")
display(df.groupby("platform")["text_length"].describe().round(0))

In [ ]:
# Volume patterns: time of day and day of week
df_with_time = df.dropna(subset=["timestamp"]).copy()
df_with_time["hour"] = df_with_time["timestamp"].dt.hour
df_with_time["day_of_week"] = df_with_time["timestamp"].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posts by hour
hour_counts = df_with_time.groupby(["platform", "hour"]).size().unstack(level=0, fill_value=0)
hour_counts.plot(kind="line", ax=axes[0], marker=".")
axes[0].set_title("Post Volume by Hour of Day")
axes[0].set_xlabel("Hour (UTC)")
axes[0].set_ylabel("Posts")

# Posts by day of week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_counts = (
    df_with_time.groupby(["platform", "day_of_week"])
    .size()
    .unstack(level=0, fill_value=0)
    .reindex(day_order)
)
day_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Post Volume by Day of Week")
axes[1].set_ylabel("Posts")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Platform comparison summary table
comparison = (
    df.groupby("platform")
    .agg(
        post_count=("id", "count"),
        unique_sources=("source", "nunique"),
        avg_text_length=("text_length", "mean"),
        median_text_length=("text_length", "median"),
        avg_word_count=("word_count", "mean"),
        unique_authors=("author", "nunique"),
    )
    .round(1)
    .sort_values("post_count", ascending=False)
)
print("Platform comparison:")
display(comparison)

## Data Quality Assessment

In [ ]:
# Missing fields per source
print("Missing values per column:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
display(missing_report[missing_report["missing_count"] > 0])

# Missing by platform
print("\nMissing 'timestamp' by platform:")
print(df.groupby("platform")["timestamp"].apply(lambda x: x.isnull().sum()))

In [ ]:
# Duplicate detection
n_duplicates = df.duplicated(subset=["text"], keep="first").sum()
print(f"Exact text duplicates: {n_duplicates} ({n_duplicates / len(df) * 100:.1f}%)")

if n_duplicates > 0:
    dup_by_platform = df[df.duplicated(subset=["text"], keep=False)].groupby("platform").size()
    print("\nDuplicates by platform:")
    print(dup_by_platform.to_string())

In [ ]:
# Language filtering -- keep Spanish only
# TODO: Apply langdetect when running on real data
# from langdetect import detect
# df["detected_lang"] = df["text"].apply(lambda t: detect(t) if len(t) > 20 else "unknown")
# non_spanish = df[df["detected_lang"] != "es"]
# print(f"Non-Spanish posts to filter: {len(non_spanish)}")

print("Language filtering: TODO -- apply langdetect on real collected data")
print("Expected: >95% Spanish content from Colombian sources")

In [ ]:
# Save cleaned data to intermediate layer
# Remove exact duplicates and posts with empty text
df_clean = df.drop_duplicates(subset=["text"], keep="first")
df_clean = df_clean[df_clean["text"].str.len() > 10]  # Remove very short/empty posts

print(f"Before cleaning: {len(df)} posts")
print(f"After cleaning:  {len(df_clean)} posts")
print(f"Removed: {len(df) - len(df_clean)} posts")

intermediate_dir = Path("../../data/02_intermediate")
intermediate_dir.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(intermediate_dir / "all_posts_cleaned.parquet", index=False)
print(f"\nSaved cleaned data to {intermediate_dir / 'all_posts_cleaned.parquet'}")

## Key Findings

**Which platforms have the most political content?**
- RSS feeds provide the highest volume of structured political news
- GDELT contributes international media perspective with tone scores

**Surprising patterns:**
- Content length varies significantly by platform (RSS articles vs GDELT summaries)
- Some articles appear in both RSS and GDELT collections -- deduplication is important

**Data gaps to be aware of:**
- ACLED has reporting lag; recent events may be underrepresented
- Weekend volume may drop for institutional sources (government, news outlets)

## Notes & Next Steps

1. **Language filtering** -- apply `langdetect` to remove non-Spanish content
2. **Temporal alignment** -- normalize all timestamps to Colombia timezone (UTC-5)
3. **Feed into analysis pipeline** -- sentiment (notebook 3-analysis/01), NER (02), topics (03)
4. **Platform-specific NLP** -- different platforms may benefit from different preprocessing

## References

- Seaborn statistical data visualization: https://seaborn.pydata.org/
- DuckDB: https://duckdb.org/
- Pandas time series analysis: https://pandas.pydata.org/docs/user_guide/timeseries.html